# Query planning in QuackWeave

**Architecture proposal grounded in QuackWeave `f45c2c1` and DuckDB 2.0.1 `8d78114e90`.** Give DuckDB a typed graph scan that it can plan alongside its other relations. Give the selected graph engine a read that it can plan and execute using its own indexes, traversal algorithms and storage. The integration decision is which SQL operations may cross that scan boundary without changing observable results.

For a standalone read, the natural delegation unit is the entire admitted graph statement. For SQL/PGQ, it is the graph pattern and required properties. DuckDB retains the surrounding relational expressions, joins, grouping, ordering and limits unless a specific rewrite establishes equivalence. The public entry points remain bare `MATCH`, `GRAPH_TABLE` and the explicit-schema `qw_query` function. [High-level architecture](quack-weave-architecture.md) · [Usage specification](quack-weave-hybrid-usage-spec.ipynb)

## What DuckDB separates

| Stage | Handoff in the tested host | Implication for QuackWeave |
|---|---|---|
| Parse and transform | `Parser::ParseQuery` produces statements and parsed expressions. PEG grammar extensions can lower new syntax into existing statement and table-reference forms. | Preserve graph source spans, variable names and parameter locations. Let DuckDB parse the enclosing SQL. |
| Bind and build a logical plan | `Planner::CreatePlan` invokes the binder. A table-function bind resolves a result schema so parents can reference typed columns by binding. | Resolve graph alias, catalog, generation, property types, parameters and output slots here. Graph registration can discover metadata. Query binding should avoid scanning graph data. |
| Optimize the logical plan | `Optimizer::Optimize` runs extension pre-hooks, built-in passes, then extension post-hooks. Built-ins include expression rewriting, filter pushdown, join ordering, column pruning, statistics propagation and join-filter pushdown. | Expose relational operators and truthful scan capabilities. The order matters. A rewrite introduced in a post-hook does not automatically receive a second run of the built-ins. |
| Create a physical plan | `PhysicalPlanGenerator::CreatePlan(LogicalGet &)` can produce a `PhysicalTableScan`. It accounts for filter columns, projected column IDs and whether the scan actually supports projection pushdown. | Retain the ordinary physical scan and implement the exact column contract. A capability flag is a promise about rows and slots. |
| Execute | `Executor::InitializeInternal` builds and schedules pipelines. Sources feed chunks to DuckDB operators, including the probe side of a hash join. | Acquire the provider reader at execution, after generation/access checks. Build-side watch keys become available at execution. Keep mutable reader and parameter state per execution. |

I used `code_symbol_search`, `code_read_symbol` and `code_callees` on the registered **duckdb** project. The project checkout and release pin differ. I checked the stage order and extension hooks against the pinned source. The optimizer call sequence is visible in function bodies even where the graph index leaves C++ call edges unresolved. [Parser](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/parser/parser.cpp#L253) · [Preparation](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/main/client_context.cpp#L317) · [Binder](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/planner/planner.cpp#L213) · [Optimizer hooks](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/optimizer/optimizer.cpp#L592) · [Scan planning](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/execution/physical_plan/plan_get.cpp#L31) · [Pipeline construction](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/parallel/executor.cpp#L270)

### From parsing to the first graph batch

The diagram shows one successful execution. The physical planner is a separate DuckDB stage. The provider prepares or plans the graph fragment inside the final call. Its internal plan is opaque to DuckDB. The sequence check verifies an ordered trace of these messages. The live plans below exercise the actual extension.

In [ ]:
sequenceDiagram
    participant Grammar
    participant Binder
    participant Optimizer
    participant PhysicalPlanner
    participant Executor
    participant Provider
    Note over Grammar,Provider: @spec QUERY-LIFECYCLE
    Grammar->>Binder: parsed
    Note over Grammar,Binder: @message E1<br/>@from Grammar<br/>@to Binder<br/>@event parsed<br/>@order 1<br/>@when true<br/>@ensures STEP1: true
    Binder->>Optimizer: logical
    Note over Binder,Optimizer: @message E2<br/>@from Binder<br/>@to Optimizer<br/>@event logical<br/>@order 2<br/>@when true<br/>@ensures STEP2: true
    Optimizer->>PhysicalPlanner: optimized_logical
    Note over Optimizer,PhysicalPlanner: @message E3<br/>@from Optimizer<br/>@to PhysicalPlanner<br/>@event optimized_logical<br/>@order 3<br/>@when true<br/>@ensures LOGICAL_PLAN: true
    PhysicalPlanner->>Executor: physical_ready
    Note over PhysicalPlanner,Executor: @message E6<br/>@from PhysicalPlanner<br/>@to Executor<br/>@event physical_ready<br/>@order 4<br/>@when true<br/>@ensures PHYSICAL_PLAN: true
    Executor->>Provider: execute_fragment
    Note over Executor,Provider: @message E4<br/>@from Executor<br/>@to Provider<br/>@event execute_fragment<br/>@order 5<br/>@when true<br/>@ensures STEP4: true
    Provider->>Executor: return_batch
    Note over Provider,Executor: @message E5<br/>@from Provider<br/>@to Executor<br/>@event return_batch<br/>@order 6<br/>@when true<br/>@ensures STEP5: true
    Note over Grammar,Provider: @verify ORDER: prove sequence_protocol

## Where the current implementation crosses stages

| Entry or rule | What the code does today | Architectural consequence |
|---|---|---|
| Bare GQL or Cypher | `TransformMatch` carries the authored source and session settings into a table reference. Named parameters travel as values. The GQL frontend parses and lowers the supported read subset to Cypher for these providers. Cypher uses provider-specific read admission. | Keep an opaque, provider-owned read for standalone execution. Preserve result names, types, row multiplicity, NULLs and any specified ordering. List dialect and result-type coverage per provider. Ladybug has typed graph values, while the current RelateDB interface exposes supported scalar projections. |
| `GRAPH_TABLE` | `ParsedGraphTable::Lower` constructs a host `SelectNode`, collects graph properties and emits a Cypher projection. It also inspects SQL filters and join syntax to select a filtered or watchlist function. | Surface projections and SQL expressions compose well. A grammar transform choosing an execution strategy ties optimization to spelling, such as `ON` versus `USING`, and obscures equivalent queries through a view. |
| Aggregate pushdown | The pre-optimizer walks a bound aggregate/projection/get chain. It reconstructs provenance by checking generated `MATCH … RETURN …` text and aliases, rebinds a replacement `qw_query`, and checks registration generation. | Keep the bound logical rule. Give it validated property bindings so a formatting change cannot alter eligibility. Preserve the generation, quota and type guards. |
| Runtime watchlist | The post-hook recognizes an eligible optimized inner join, invokes DuckDB’s join-filter pass for the graph scan and bridges dynamic keys through local initialization. It temporarily adjusts scan flags for an identity projection. | Keep DuckDB’s join and build-side timing. Isolate this version-sensitive C++ bridge. General scan projection support must preserve reordered slots, filter-only columns and zero-column reads before the flags can be enabled broadly. |
| Provider execution | Rust `GraphQuery` binds and manages the reader. Ladybug uses its native prepared statement and Arrow batch path. RelateDB executes on a fresh connection to the registered frozen handle, materializes the native result, then batches it for DuckDB. | Keep the shared scan lifecycle; specify batch, memory and timeout behavior separately for each provider. A DuckDB chunk interface alone cannot make RelateDB stream an unbounded result. |

[Standalone transform](crates/native/grammar/standalone.cpp) · [PGQ lowering](crates/native/grammar/graph_table.cpp) · [Aggregate rule](crates/native/grammar/aggregate_pushdown.cpp) · [Runtime filter bridge](crates/native/grammar/runtime_watchlist.cpp) · [Bound scan and execution](crates/core/src/qw_query.rs) · [Ladybug](crates/native/src/ladybug.rs) · [RelateDB](crates/native/src/relate_backend.rs)

## Proposed ownership and interfaces

**1. Frontend and binding.** Keep two fragment forms. An opaque standalone fragment holds the authored dialect, text/span and parameter slots. A structured SQL/PGQ fragment holds the supported fixed pattern, variable identity, entity kind, label/type and graph-property references. Reuse the existing GQL parse representation where it applies. Each fragment binds to one provider, graph generation, ordered result schema and explicit graph-to-SQL column mapping. Preserve SQL aliases separately from case-sensitive graph property names. The binder must validate any descriptor reachable through a callable table function. A caller-supplied string or flag cannot certify a safe rewrite.

**2. One DuckDB logical scan per graph fragment.** Build on `BoundRead` and the existing table functions. Store validated projected slots, provider capabilities and optional bound predicates with the scan. Keep the bound description reusable across prepared executions. Put actual watch keys, native connections, cancellation state and batch cursors in execution-local state. A SQL join between two providers can have two independent scans; DuckDB owns that join. Distinct providers have distinct snapshots.

**3. Let DuckDB offer work to the scan.** Use the existing filter callback for eligible bound expressions, even when a predicate originated outside `GRAPH_TABLE`. Map the bound column index back to the graph property. Distinguish an optional prefilter that retains the SQL residual from exact enforcement that allows DuckDB to remove it. The Rust `rusty_sequence` example already demonstrates `pushdown_filter`, the index mapping and `accept_pushdown`. Extend projection pruning only for structured reads with safe property projections. Pruning an expression from an opaque native return list could suppress a conversion error, change distinctness or ordering. Supply row estimates independently of correctness bounds. The current watch hint of 65 is derived from a key threshold; a single key can produce thousands of graph rows.

**4. Keep graph-specific optimizer rules small.** One rule may use a DuckDB hash join’s exact keys to restrict a graph scan. Another may replace a proven scalar aggregate with a provider aggregate. The rule owns eligibility and the DuckDB residual. The provider owns translation to the backend dialect and native execution. Start with the pinned pre-/post-hook placements above. Move a rule to a different phase only after plan tests confirm which filters, bindings and projections are available there. [Bound filter example](crates/core/src/sequence.rs) · [Table-function callbacks](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/include/duckdb/function/table_function.hpp#L493) · [DuckDB built-in optimizer passes](https://github.com/duckdb/duckdb/blob/8d78114e90a53823803660bf821f6073ac87da8c/src/optimizer/optimizer.cpp#L230)

### Pushdown decisions

| Operation | Evidence needed | DuckDB work that remains |
|---|---|---|
| Typed equality or a small integer `IN` | Direct bound property with compatible type, NULL and comparison semantics. A declared RelateDB type alone cannot rule out a stored value of another type. Respect configured execution limits. | Keep the residual unless enforcement and observable errors are equivalent. Unsupported expressions run as ordinary SQL filters. |
| Watchlist supplied by a table | Eligible inner equijoin, supported key type, exact keys from the current build side and the same graph generation. Today the bridge accepts up to 64 keys. | Retain the entire SQL join. A watchlist containing `(7),(7),(NULL)` must return each graph match for 7 twice. Oversized, missing or unusable dynamic filters fall back to the base scan. |
| Ungrouped scalar count or sum | Check the aggregate, argument types, filters, distinctness, NULL-on-empty behavior, overflow and resource settings. On RelateDB a frozen single-node or directed single-edge row bound can prove a positive-modulus sum fits its signed accumulator. | Keep grouped, unsafe or resource-limited cases in DuckDB. Preserve SQL `BIGINT` count and `HUGEINT` sum types. |
| SQL outer join, order, limit, or a second provider | A future rewrite would need separate evidence for null-extension, collation, ordering, cardinality and snapshot semantics. | Use DuckDB’s normal operators. A limit in an authored standalone graph statement remains part of that graph statement. |

[Watchlist fixture](test/relate/table_watch.test) · [Predicate fixture](test/relate/predicate_pushdown.test) · [Aggregate and fallback fixtures](test/relate/aggregate.test)

### Two statements, two delegation boundaries

With the graph selected and the dialect set to GQL or Cypher, this standalone read delegates the pattern, property predicate and return projection to the graph engine. The result column is `id`.

```sql
SESSION SET GRAPH social;
MATCH (m:Message)-[r:Message_hasCreator_Person]->(p:Person)
WHERE p.PersonId = 4398046519825
RETURN m.MessageId AS id;
```

With the benchmark's five-row DuckDB watch table, this hybrid statement lets the graph engine traverse the same pattern. DuckDB owns the join to the live table, duplicate handling, grouping, checksum and ordering. It can send eligible IDs to the graph scan once the join’s build side has them.

```sql
SELECT g.author, count(*), sum(g.mid % 1000003)
FROM GRAPH_TABLE (
    social MATCH (m:Message)-[r:Message_hasCreator_Person]->(p:Person)
    COLUMNS (p.PersonId AS author, m.MessageId AS mid)
) AS g
JOIN watch USING (author)
GROUP BY g.author
ORDER BY g.author;
```

GQL, Cypher and SQL/PGQ have separate language contracts. Keep a feature matrix for the accepted subset and result shapes of each backend. Reject an unsupported construct explicitly. A future provider that accepts the required GQL semantics directly can take the original GQL read after admission; the two present adapters use the supported GQL-to-Cypher lowering.

## Live plan and result checks

The next cell loads the local combined extension into the benchmark’s pinned DuckDB 2.0.1 through ADBC and uses the existing modulus-256 fixture, with 14,883 creator edges. It starts each provider independently, executes each statement, compares the result with the CSV reference, and inspects the physical graph scan. For standalone syntax it uses `PREPARE … AS MATCH …` followed by `EXPLAIN (ANALYZE, FORMAT JSON) EXECUTE …`. The pinned grammar rejects the direct `EXPLAIN … MATCH` spelling. The “selected DuckDB operators” column lists operators relevant to the boundary; the stored plan also contains execution wrappers.

The final case sets a 1 MiB conversion budget. The identical SQL then takes the ordinary graph scan and host aggregate path. Both numeric results must match. To replay on a fresh checkout, provision the combined extension, ADBC environment and modulus-256 fixture as described in the benchmark instructions. [Fixture and setup](benchmarks/provider_comparison/README.md) · [Workload SQL and reference](benchmarks/provider_comparison/run.py)

In [3]:
from pathlib import Path
import json
import subprocess
from IPython.display import Markdown, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "benchmarks/provider_comparison/run.py").is_file())
# The repository's ADBC worker loads its pinned DuckDB 2.0.1 library and enforces the source ID.
worker = "from pathlib import Path\nfrom types import SimpleNamespace\nimport json\nimport sys\n\nroot = Path.cwd()\nsys.path.insert(0, str(root / \"benchmarks\" / \"provider_comparison\"))\nimport run\n\nfixture = run.BUILD / \"mod256\"\nreference = run.oracle(fixture)\n\ndef operators(nodes):\n    for node in nodes:\n        yield node\n        yield from operators(node.get(\"children\", []))\n\nrows = []\nfor provider in run.ENGINES:\n    args = SimpleNamespace(provider=provider, fixture=fixture, extension=run.EXT, library=run.LIB)\n    with run.cursor(args) as (cur, _):\n        for case in (\"gql_lookup\", \"cypher_fanout\", \"graph_table_where\", \"table_watchlist\", \"full_scan_aggregate\", \"budget_fallback\"):\n            key = \"full_scan_aggregate\" if case == \"budget_fallback\" else case\n            dialect, sql = run.CASES[key]\n            cur.execute(\"SET SESSION qw_graph_dialect = \" + run.sql_literal(dialect))\n            if case == \"budget_fallback\":\n                config = {\"alias\": \"social\", \"path\": str((fixture / run.ENGINES[provider]).resolve()),\n                          \"max_batch_bytes\": 1048576}\n                cur.execute(f\"SET GLOBAL qw_{provider}_config = \" + run.sql_literal(json.dumps(config)))\n            # PREPARE accepts the standalone grammar even when EXPLAIN expects a SQL statement.\n            cur.execute(\"PREPARE inspect_graph AS \" + sql)\n            cur.execute(\"EXPLAIN (ANALYZE, FORMAT JSON) EXECUTE inspect_graph\")\n            plan = list(operators(json.loads(cur.fetchone()[1])[\"operator\"]))\n            answer, _ = run.fetch(cur, \"EXECUTE inspect_graph\")\n            assert answer == reference[\"expected\"][key], (provider, case, answer)\n            scan, = [node for node in plan if node[\"name\"].startswith(\"QW_\")]\n            expected_rows = {\"gql_lookup\": 1, \"cypher_fanout\": reference[\"fanout\"],\n                             \"graph_table_where\": reference[\"fanout\"],\n                             \"table_watchlist\": sum(row[1] for row in reference[\"watchlist_groups\"]),\n                             \"full_scan_aggregate\": 1, \"budget_fallback\": reference[\"edges\"]}[case]\n            assert scan[\"intermediate_rows\"] == expected_rows\n            names = [node[\"name\"] for node in plan]\n            if case == \"table_watchlist\":\n                assert \"HASH_JOIN\" in names and \"HASH_GROUP_BY\" in names\n            if case == \"budget_fallback\":\n                assert \"UNGROUPED_AGGREGATE\" in names\n            rows.append([provider, case, scan[\"name\"], expected_rows,\n                         \", \".join(n for n in names if n in (\"HASH_JOIN\", \"HASH_GROUP_BY\", \"UNGROUPED_AGGREGATE\", \"FILTER\", \"ORDER_BY\")) or \"—\"])\nprint(json.dumps(rows))"
completed = subprocess.run([str(root / "build/usage-client-venv/bin/python"), "-B", "-c", worker],
                           cwd=root, capture_output=True, text=True)
if completed.returncode:
    raise RuntimeError(completed.stderr)
rows = json.loads(completed.stdout)
assert len(rows) == 12
display(Markdown("| Provider | Query | Graph scan | Rows from graph | Selected DuckDB operators |\n"
                 "|---|---|---|---:|---|\n" +
                 "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))

| Provider | Query | Graph scan | Rows from graph | Selected DuckDB operators |
|---|---|---|---:|---|
| ladybug | gql_lookup | QW_GRAPH_QUERY | 1 | — |
| ladybug | cypher_fanout | QW_GRAPH_QUERY | 22 | — |
| ladybug | graph_table_where | QW_GRAPH_QUERY | 22 | FILTER |
| ladybug | table_watchlist | QW_GRAPH_QUERY | 82 | ORDER_BY, HASH_GROUP_BY, HASH_JOIN |
| ladybug | full_scan_aggregate | QW_QUERY | 1 | — |
| ladybug | budget_fallback | QW_GRAPH_QUERY | 14883 | UNGROUPED_AGGREGATE |
| relate | gql_lookup | QW_GRAPH_QUERY | 1 | — |
| relate | cypher_fanout | QW_GRAPH_QUERY | 22 | — |
| relate | graph_table_where | QW_GRAPH_QUERY | 22 | FILTER |
| relate | table_watchlist | QW_GRAPH_QUERY | 82 | ORDER_BY, HASH_GROUP_BY, HASH_JOIN |
| relate | full_scan_aggregate | QW_QUERY | 1 | — |
| relate | budget_fallback | QW_GRAPH_QUERY | 14883 | UNGROUPED_AGGREGATE |

## Migration and acceptance

1. **Introduce bound fragment metadata behind the existing public entry points.** Keep the current execution path as the reference. Check identical types and rows for bare GQL, bare Cypher, `GRAPH_TABLE` and explicit schema reads on both providers. Cover parameters, escaped literals, graph property case, alias collisions, same-named node and edge types, and generation changes between prepare and execute.
2. **Move static predicate selection from syntax matching to bound scan callbacks.** Compare inner and outer SQL spellings and a predicate through a view. Assert equal rows and errors with pushdown enabled and bypassed; check provider rows separately. Add projection mapping tests for reordered, repeated, unused, filter-only and zero projected columns.
3. **Migrate runtime-watch and aggregate rules one at a time.** Keep each old execution path until its replacement passes SQLLogic for `ON`/`USING`, duplicates, NULLs, empty and oversized lists, outer joins, wide sums, explicit budgets and unsupported path shapes. Assert retained host operators as well as provider row counts. Remove the redundant grammar/text recognizer only once equivalent forms are covered.
4. **Expose and benchmark the physical boundary.** Show the provider, delegation decision, retained residual and reason for fallback in plan diagnostics without logging parameter values. Benchmark cold registration, warm p50/p95, peak RSS and rows crossing the graph scan. The full SF1 baseline for the optimized release is 6,465 rows for equality, 23,316 for the table watchlist and one for the scalar aggregate on both providers. Keep this cardinality check distinct from latency and run old/new binaries on the same host. [Cross-provider SQLLogic](test/relate/benchmark_contract.test) · [Recorded SF1 observations](benchmarks/provider_comparison/optimized/mod1.json)

**First implementation slice:** carry one bound integer property from a SQL/PGQ pattern into a DuckDB filter callback; demonstrate identical results and reduced provider rows for a graph-local predicate and an outer SQL predicate. This exercises the central interface without changing the native standalone query path.

## Solver check of the proposed routing policy

Jev 1.14.0 selected `configuration.requires_any` in verify mode for the dependency between an optional watch prefilter and the retained SQL join (open gate, weakest confidence 0.75). I added five explicit dependencies from the same catalog rule. A watch prefilter also requires a verified key and a bound generation; a graph scan requires native traversal; an aggregate replacement requires equivalent aggregate semantics and permission under the configured limits. Each dependency is a separate rule, so all applicable requirements must hold.

| Declared configuration | Z3 result | Persisted solve |
|---|---|---|
| Standalone graph scan with native traversal | SAT, pass | `sol_dd486183bbb84ad7` |
| Graph scan with watch prefilter and SQL join | SAT, pass | `sol_2764ebbd6e7d45ae` |
| Graph scan with a semantically verified, allowed aggregate | SAT, pass | `sol_793ce7de664c4238` |
| Same watch prefilter with the host join removed | UNSAT, fail | `sol_e7fc447839724de8` |
| Aggregate replacement without semantic equivalence | UNSAT, fail | `sol_c6272d15a35f4495` |

These results verify the declared ownership policy. The live experiment establishes the operator placement for the tested statements. SQLLogic and differential benchmarks remain the acceptance evidence for each future rewrite.

## Implementation and verification

Standalone GQL and Cypher keep their native read path. For a hybrid query, the grammar supplies a versioned description of the graph pattern. The binder checks it against the native read and records the property behind each DuckDB output column. DuckDB can then offer projected column IDs, bound integer predicates and execution-local hash-join keys. The provider traverses the graph. DuckDB keeps the SQL filter and join. A scalar aggregate moves only when the rule verifies its type, arithmetic and resource limits.

| Task | Commit | Fixture-backed evidence |
|---|---|---|
| Bind the graph fragment | `60df489` | Validate slot order, generated aliases and the native read before planning a rewrite. |
| Project requested properties | `2e45282` | Reordered, repeated, unused, filter-only and constant projections retain their result types. |
| Place scalar aggregates | `ab212ed` | A matching authored query keeps its host aggregate. The structured PGQ form can return one aggregate row. |
| Apply static SQL predicates | `fa2cc0a` | A filter in graph syntax, outer SQL or a view reaches the same bound integer slot. DuckDB retains the residual. |
| Explain the scan | `37c350f` | The graph scan identifies its provider, operation, residual or fallback when applicable. DuckDB can separately include submitted SQL in the profile. |
| Estimate the graph scan | `57f3ec9` | Fixed vertex and one-step scans use registered populations. RelateDB's global counts can exceed a label's actual row count. |
| Read join keys at execution | `d1541e6` | DuckDB can filter the graph through a renamed view when it builds the hash table on the watchlist. The same SQL bag is returned when the join filter is disabled. |
| Preserve explicit limits for legacy watchlists | `a6472d5` | A nonmatching literal or runtime key still raises the configured batch error. A timeout-limited graph reads the original input. |
| Match Ladybug's catalog casing | `4e3d948` | A mixed-case one-step Ladybug scan receives the same estimate as the catalog spelling. RelateDB keeps exact-case binding. |
| Explain a host-owned comparison join | `1ef6b47` | A preserved outer-join input reports the retained SQL join and the absence of an eligible runtime filter. |

The live DuckDB cell above was replayed on the combined production extension built from `1ef6b47`. Its smaller fixture yielded 82 watchlist rows for each provider. The delegated scalar aggregate returned one row. An explicit conversion budget retained all 14,883 input rows for the DuckDB aggregate. The matched full-size SF1 report measures the same boundary on 3,809,607 edges.

### Per-task solver records

The identifiers below address the persisted Z3 checks. For the runtime watch, retaining the SQL join passed on both revisions. Its failing SQLLogic case measured excess rows crossing the provider boundary.

| Task | Before | After |
|---|---|---|
| Bound fragments | UNSAT `sol_1838ace3c38a434d` | SAT `sol_52d736ff7e5a4b8c` |
| Projection | UNSAT `sol_f10af08e11014c30` | SAT `sol_58bb52fa25fe4257` |
| Scalar aggregate | UNSAT `sol_972bec2f966a4c54` | SAT `sol_fb291d05b6e04f7f` |
| Static SQL predicate | UNSAT `sol_4f606a2239614bbd` | SAT `sol_4749f88d0cd949f9` |
| Runtime watch | SAT `sol_1018e65b109c415c` | SAT `sol_b027a35e71474ba9` |
| Scan diagnostics | UNSAT `sol_012830305eb04f4c` | SAT `sol_60cfb1d50ecc4cff` |
| Graph population estimates | UNSAT `sol_fbae63dddf8e4607` | SAT `sol_9451d92a74bd490a` |
| Legacy watchlist budgets | UNSAT `sol_6186d96a9b5d4137` | SAT `sol_a12e6584c3ef4faf` |
| Mixed-case Ladybug estimates | UNSAT `sol_d97f1ae23e034849` | SAT `sol_35c8f0ac19484361` |
| Ordinary host-join diagnostic | UNSAT `sol_a7ba04785c594076` | SAT `sol_9f5f1ea3e65342b0` |
| Cross-provider acceptance | UNSAT `sol_32505f7ae0f34c72` | SAT `sol_ca7bfd55763641d3` |

The tasks ran Jev before and after the change. For the fragment post-check, aggregate checks, host-join diagnostic checks and some acceptance checks, Jev's confidence gate blocked automatic routing. Those gates stayed blocked. The reviewed catalog constraints were checked separately with Z3. The solver checks verify the declared planning invariants. The fixture-backed SQLLogic cases check the actual DuckDB and provider behavior. [Cross-provider SQLLogic cases](test/relate) · [Ladybug stabilization cases](test/stabilization/sqllogic) · [Matched SF1 measurements and reproduction](benchmarks/provider_comparison/planning/README.md)

## Ladybug point lookup returning zero rows

The Linux arm64 benchmark at `7bdfb95` measured 0.325 ms for direct Cypher and 1.769 ms for bare `MATCH` through DuckDB. Both returned zero rows. Run the Rust cells below in order using the project configuration `configure`. They compare client preparation, the binder’s use of session settings, and a candidate for reusing Ladybug’s read-admission decision. The live measurements use the local SF1 creator graph and report their own timings.

The fixture, Python benchmark environment and release extension come from [the provider comparison setup](benchmarks/provider_comparison/README.md). [Archived CI measurements](https://github.com/kevintruong/quack-weave/tree/benchmark-history/main/7bdfb95c951801779130e57c426bb7b53a7c453e/45551464093-1)

In [11]:
// configure
use std::{env, fs, path::{Path, PathBuf}, process::Command};
use rusty_quack_core::qw_query::GraphGeneration;

fn qw_checkout() -> PathBuf {
    let start = env::var_os("QW_PROJECT_ROOT").map(PathBuf::from)
        .unwrap_or_else(|| env::current_dir().expect("Rust kernel working directory"));
    start.ancestors().find(|p| p.join("benchmarks/provider_comparison/tracker.py").is_file())
        .expect("Open the notebook in the Quack Weave checkout or set QW_PROJECT_ROOT").to_owned()
}
let qw_root = qw_checkout();
let fixture = qw_root.join("build/perf/provider-comparison/mod1");
let graph_file = fixture.join("social.lbdb");
let bench_python = qw_root.join("build/usage-client-venv/bin/python");
let extension = env::var_os("QW_EXTENSION").map(PathBuf::from).unwrap_or_else(||
    qw_root.join("build/native/release/extension/quack_weave/quack_weave.duckdb_extension"));
for input in [&graph_file, &bench_python, &extension] {
    assert!(input.is_file(), "Missing benchmark input {}", input.display());
}
let generation = GraphGeneration::capture(1, &graph_file).expect("capture fixture identity");
generation.validate(&graph_file).expect("verify fixture identity");
println!("configure • rusty_quack_core • SF1/mod1 • {} bytes\nExtension {}", 
    fs::metadata(&graph_file)?.len(), extension.display());

configure • rusty_quack_core • SF1/mod1 • 198979584 bytes


Extension /Users/kevintruong/project/quack-weave/build/mvp2-relate/release/extension/quack_weave/quack_weave.duckdb_extension


### What preparing a read changes

The tracker runs the same hit, miss and fanout in mixed order through the native client and the extension. The next control holds DuckDB preparation and the inferred projection constant. Only the graph alias and dialect arguments differ. It alternates batches within one pinned CLI process and checks that every execution emits zero rows. Its batch times include SQL `EXECUTE` dispatch. The last cell in this group checks what happens to an existing statement when the session, registration or file permissions change.

In [12]:
// Measure the public clients. The tracker owns the fixture oracle, warm-up and Arrow fetch.
use std::time::{SystemTime, UNIX_EPOCH};
let report_dir = qw_root.join("build/notebook-ladybug-miss");
fs::create_dir_all(&report_dir)?;
let nonce = SystemTime::now().duration_since(UNIX_EPOCH)?.as_nanos();
let report_path = report_dir.join(format!("tracker-{nonce}.json"));
let run = Command::new(&bench_python).current_dir(&qw_root)
    .args(["-B", "benchmarks/provider_comparison/tracker.py", "run", "--provider", "ladybug",
           "--case", "lookup", "--case", "lookup_miss", "--case", "fanout",
           "--include-prepared", "--blocks", "2", "--runs", "20", "--fixture"])
    .arg(&fixture).arg("--extension").arg(&extension).arg("--out").arg(&report_path).output()?;
assert!(run.status.success(), "{}", String::from_utf8_lossy(&run.stderr));
let snapshot: serde_json::Value = serde_json::from_slice(&fs::read(&report_path)?)?;
assert_eq!(snapshot["expected"]["lookup_miss"][0].as_u64(), Some(0));
let miss = snapshot["statements"]["lookup_miss"].as_str().unwrap().to_owned();
let miss_rows: Vec<_> = snapshot["summary"].as_array().unwrap().iter()
    .filter(|r| r["case"] == "lookup_miss").collect();
assert_eq!(miss_rows.len(), 4);
println!("{} • {} • verified empty result\n{:<24} {:>4} {:>9} {:>9}", 
    snapshot["platform"], snapshot["versions"]["ladybug"], "route", "n", "p50 ms", "p95 ms");
for r in miss_rows {
    println!("{:<24} {:>4} {:>9.3} {:>9.3}", r["route"].as_str().unwrap(), r["n"],
        r["p50_ms"].as_f64().unwrap(), r["p95_ms"].as_f64().unwrap());
}
println!("Raw samples and binary hashes: {}", report_path.display());

"macOS-26.7.1-arm64-arm-64bit-Mach-O" • "0.22.3" • verified empty result


route                       n    p50 ms    p95 ms


direct_cypher            40     0.091     0.168


qw_cypher                40     0.520     0.692


qw_gql                   40     0.531     0.706


qw_prepared_cypher       40     0.224     0.281


Raw samples and binary hashes: /Users/kevintruong/project/quack-weave/build/notebook-ladybug-miss/tracker-1795588041512251000.json


In [13]:
// Isolate binding with identical inferred projections and prepared handles.
use std::{io::Write, process::{Output, Stdio}};
fn sql_text(value: &str) -> String { format!("'{}'", value.replace('\'', "''")) }
fn duck_script(cli: &Path, script: &str) -> std::io::Result<Output> {
    let mut child = Command::new(cli).args(["-no-agent", "-unsigned", "-batch", "-noheader", "-list"])
        .stdin(Stdio::piped()).stdout(Stdio::piped()).stderr(Stdio::piped()).spawn()?;
    child.stdin.take().unwrap().write_all(script.as_bytes())?;
    child.wait_with_output()
}
let cli = qw_root.join("build/deps/duckdb-v2.0.1/duckdb");
let registration = serde_json::json!({"alias":"social", "path":graph_file});
let setup_sql = format!(".bail on\nSET threads=1;\nLOAD {};\nSET GLOBAL qw_ladybug_config={};\nSESSION SET GRAPH social;\nSET SESSION qw_graph_dialect='cypher';\n",
    sql_text(extension.to_str().unwrap()), sql_text(&registration.to_string()));
let query_arg = sql_text(&miss);
let plans = format!("PREPARE live_q AS SELECT * FROM qw_graph_query(current_setting('qw_selected_graph'),current_setting('qw_graph_dialect'),{query_arg});\nPREPARE fixed_q AS SELECT * FROM qw_graph_query('social','cypher',{query_arg});\n");
let mut script = format!("{setup_sql}{plans}");
let mut samples = std::collections::BTreeMap::<String, Vec<f64>>::new();
let mut order = Vec::new();
let batch = 32;
for block in 0..12 {
    let routes = if block % 2 == 0 { ["live_q", "fixed_q"] } else { ["fixed_q", "live_q"] };
    for route in routes {
        order.push((block, route));
        script.push_str("SELECT 'clock|' || epoch_us(current_timestamp);\n");
        script.push_str(&format!("EXECUTE {route};\n").repeat(batch));
        script.push_str("SELECT 'clock|' || epoch_us(current_timestamp);\n");
    }
}
let measured = duck_script(&cli, &script)?;
assert!(measured.status.success(), "{}", String::from_utf8_lossy(&measured.stderr));
let stdout = String::from_utf8(measured.stdout)?;
let clocks: Vec<i64> = stdout.lines().map(|line|
    line.strip_prefix("clock|").expect("lookup must emit zero rows").parse().unwrap()).collect();
assert_eq!(clocks.len(), order.len() * 2);
for ((block, route), t) in order.into_iter().zip(clocks.chunks_exact(2)) {
    if block >= 2 {
        samples.entry(route.into()).or_default().push((t[1]-t[0]) as f64 / batch as f64);
    }
}
fn median_us(v: &[f64]) -> f64 {
    let mut x = v.to_vec();
    x.sort_by(f64::total_cmp);
    (x[(x.len()-1)/2] + x[x.len()/2]) / 2.0
}
for (route, v) in &samples {
    println!("{route:8} {:.1} µs/lookup • median of {} batch means", median_us(v), v.len());
}
println!("Session settings cost {:.1} µs/lookup in this CLI control.", 
    median_us(&samples["live_q"]) - median_us(&samples["fixed_q"]));

fixed_q  251.6 µs/lookup • median of 10 batch means


live_q   462.4 µs/lookup • median of 10 batch means


Session settings cost 210.7 µs/lookup in this CLI control.


In [22]:
// Exercise the host contract on real prepared statements. Use a hit for the positive control.
let hit = snapshot["statements"]["lookup"].as_str().unwrap().to_owned();
let expected_id = snapshot["watchlist"][0].as_i64().unwrap().to_string();
let explicit = format!("PREPARE public_q AS SELECT * FROM qw_query('social','cypher',{},schema:='id BIGINT');\n", sql_text(&hit));
let selected_reset = "RESET qw_selected_graph;\nSET SESSION qw_graph_dialect='gql';\n";
let cypher_only = format!("MATCH (p:Person {{PersonId: {expected_id}}}) RETURN p.PersonId AS id");
let dialect_plan = format!("PREPARE dialect_q AS SELECT * FROM qw_graph_query(current_setting('qw_selected_graph'),current_setting('qw_graph_dialect'),{});\n", sql_text(&cypher_only));
for (name, body, expected_error) in [
    ("live statement admits a native Cypher property map", format!("{dialect_plan}EXECUTE dialect_q;"), None),
    ("live statement observes a dialect change", format!("{dialect_plan}SET SESSION qw_graph_dialect='gql';\nEXECUTE dialect_q;"), Some("QW_GQL_SUBSET")),
    ("explicit client survives session reset", format!("{explicit}{selected_reset}EXECUTE public_q;"), None),
    ("bare MATCH follows session reset", format!("{selected_reset}{hit};"), Some("QW_CONFIG")),
    ("live prepared scan follows session reset", format!("{plans}{selected_reset}EXECUTE live_q;"), Some("QW_CONFIG")),
    ("prepared scan observes registration reset", format!("{explicit}RESET GLOBAL qw_ladybug_config;\nEXECUTE public_q;"), Some("QW_CONFIG")),
    ("prepared scan observes disabled file access", format!("{explicit}SET enable_external_access=false;\nEXECUTE public_q;"), Some("QW_ACCESS")),
] {
    let result = duck_script(&cli, &format!("{setup_sql}{body}\n"))?;
    let err = String::from_utf8_lossy(&result.stderr);
    match expected_error {
        Some(code) => assert!(!result.status.success() && err.contains(code), "{name}: {err}"),
        None => assert!(result.status.success() && String::from_utf8_lossy(&result.stdout).trim() == expected_id, "{name}: {err}"),
    }
    println!("PASS  {name}");
}

PASS  live statement admits a native Cypher property map


PASS  live statement observes a dialect change


PASS  explicit client survives session reset


PASS  bare MATCH follows session reset


PASS  live prepared scan follows session reset


PASS  prepared scan observes registration reset


PASS  prepared scan observes disabled file access


()

### Which parser work can the Ladybug adapter reuse?

These experiments call the checkout’s C++ classifier and projection visitor, linked with the pinned static Ladybug archive used by the extension. The first retains the read classification for the exact Cypher text within one bound plan. The second keeps three successful projection descriptions for a registered graph generation so they can survive a DuckDB rebind. Both include their cold first lookup. Each test times a component directly. The public-client timings above measure the current extension.

In [18]:
// Build a probe for the checkout's native projection and read-admission parsers.
:dep libloading = "0.8"
use std::ffi::CStr;
use libloading::Library;
fn build_classifier(root: &Path, work: &Path) -> Result<PathBuf, Box<dyn std::error::Error>> {
    let mac = cfg!(target_os = "macos");
    let triple = if mac { "aarch64-apple-darwin" } else { "aarch64-unknown-linux-gnu" };
    let sdk = root.join("build/deps/ladybug-0.22.3");
    let src = sdk.join("ladybug-0.22.3");
    let target = env::var_os("CARGO_TARGET_DIR").map(PathBuf::from).unwrap_or_else(|| root.join("target"));
    let outputs = fs::read_dir(target.join("native").join(triple).join("release/build"))?;
    let mut headers: Vec<_> = outputs.filter_map(Result::ok).map(|e| e.path().join("out/common/system_config.h"))
        .filter(|p| p.is_file()).collect();
    headers.sort();
    let header = headers.first().ok_or("Build the combined release to generate the Ladybug storage header")?;
    let config = fs::read(header)?;
    for other in &headers[1..] { assert_eq!(fs::read(other)?, config, "Conflicting vendor configuration headers"); }
    let unit = work.join("classifier.cpp");
    fs::write(&unit, r#"#include "crates/native/src/ladybug_admission.cpp"
extern "C" __attribute__((visibility("default"))) bool probe_read(const char* text) {
    return qw_ladybug_is_graph_read(text);
}
extern "C" __attribute__((visibility("default"))) bool probe_project(const char* text, projection_field visit, void* out) {
    return qw_ladybug_project(text, visit, out);
}"#)?;
    let library = work.join(format!("classifier.{}", env::consts::DLL_EXTENSION));
    let mut cc = Command::new("c++");
    cc.args(["-std=c++20", "-O2", "-fPIC", "-fvisibility=hidden", "-ffunction-sections"]);
    cc.args(if mac { ["-dynamiclib", "-Wl,-dead_strip"] } else { ["-shared", "-Wl,--gc-sections"] });
    cc.arg("-I").arg(root).arg("-I").arg(header.parent().unwrap().parent().unwrap());
    for include in ["src/include", "third_party/alp/include", "third_party/antlr4_cypher/include",
                    "third_party/antlr4_runtime/src"] {
        cc.arg("-I").arg(src.join(include));
    }
    // Match crates/native/build.rs. The parser exposes C++ data structures across this boundary.
    let vendor = env::var_os("LADYBUG_LIB_DIR").map(PathBuf::from)
        .unwrap_or_else(|| sdk.join(if mac { "static" } else { "static-linux-aarch64" }));
    let ssl = match env::var_os("OPENSSL_LIB_DIR") {
        Some(path) => PathBuf::from(path),
        None if mac => {
            let brew = Command::new("brew").args(["--prefix", "openssl@3"]).output()?;
            if !brew.status.success() { return Err("Set OPENSSL_LIB_DIR to the OpenSSL 3 static archives".into()); }
            PathBuf::from(String::from_utf8(brew.stdout)?.trim()).join("lib")
        }
        None => return Err("Set OPENSSL_LIB_DIR to the OpenSSL 3 static archives".into()),
    };
    if mac { cc.args(["-Wl,-exported_symbol,_probe_read", "-Wl,-exported_symbol,_probe_project"]); }
    else { cc.arg("-Wl,--exclude-libs,ALL"); }
    cc.arg(unit).arg(vendor.join("liblbug.a")).arg(ssl.join("libssl.a")).arg(ssl.join("libcrypto.a"));
    if !mac { cc.args(["-ldl", "-lpthread", "-latomic"]); }
    let output = cc.arg("-o").arg(&library).output()?;
    if !output.status.success() { return Err(String::from_utf8_lossy(&output.stderr).to_string().into()); }
    Ok(library)
}
let native_work = report_dir.join(format!("native-{}", SystemTime::now().duration_since(UNIX_EPOCH)?.as_nanos()));
fs::create_dir(&native_work)?;
let probe_path = build_classifier(&qw_root, &native_work)?;
let classifier = unsafe { Library::new(&probe_path)? };
fn is_graph_read(lib: &Library, text: &CStr) -> bool {
    // The shim above declares this ABI. The borrow keeps the library live through the call.
    let classify = unsafe { lib.get::<unsafe extern "C" fn(*const std::ffi::c_char) -> bool>(b"probe_read\0") }.unwrap();
    unsafe { classify(text.as_ptr()) }
}
assert!(is_graph_read(&classifier, c"MATCH (p:Person) RETURN p"));
assert!(!is_graph_read(&classifier, c"CREATE (p:Person)"));
println!("Loaded and checked the native Ladybug read classifier.");

Loaded and checked the native Ladybug read classifier.


In [19]:
// Candidate: a Ladybug-owned read classification scoped to one bound statement.
use std::{ffi::CString, hint::black_box, time::Instant};
struct AdmittedRead(CString);
impl AdmittedRead {
    fn classify(text: &CStr, parser: &impl Fn(&CStr) -> bool) -> Option<Self> {
        parser(text).then(|| Self(text.to_owned()))
    }
    fn allows(&self, emitted: &CStr, parser: &impl Fn(&CStr) -> bool) -> bool {
        self.0.as_c_str() == emitted || parser(emitted)
    }
}
{
    // Resolve the native entry point once, outside the timer.
    let entry = unsafe { classifier.get::<unsafe extern "C" fn(*const std::ffi::c_char) -> bool>(b"probe_read\0")? };
    let parse = |s: &CStr| unsafe { entry(s.as_ptr()) };
    let original = CString::new(miss.as_str())?;
    let rewritten = CString::new(snapshot["statements"]["lookup"].as_str().unwrap())?;
    let proof = AdmittedRead::classify(&original, &parse).unwrap();
    assert!(proof.allows(&original, &parse) && proof.allows(&rewritten, &parse));
    for rejected in [c"MATCH (p:Person) DELETE p RETURN p", c"CALL show_tables() RETURN *",
                     c"MATCH (p:Person) RETURN p; CREATE (x:Person)", c"MATCH ("] {
        assert!(AdmittedRead::classify(rejected, &parse).is_none());
        assert!(!proof.allows(rejected, &parse));
    }
    println!("PASS  rewrites use native admission; updates, commands, multiple statements and malformed text rejected");
    println!("{:<26} {:>13} {:>13}", "lifetime of one plan", "current µs", "candidate µs");
    for (label, executions, alternate) in [("1 execution", 1, false), ("32 same-text executions", 32, false),
                                           ("32 alternating texts", 32, true)] {
        let mut groups = [Vec::new(), Vec::new()];
        for round in 0..30 {
            for candidate in [round % 2 == 0, round % 2 != 0] {
                let start = Instant::now();
                let receipt = if candidate { Some(AdmittedRead::classify(&original, &parse).unwrap()) } else { None };
                for i in 0..executions {
                    let emitted = black_box(if alternate && i % 2 == 1 { &rewritten } else { &original });
                    let ok = match &receipt {
                        Some(known) => known.allows(emitted, &parse),
                        None => parse(emitted),
                    };
                    assert!(black_box(ok));
                }
                drop(receipt);
                let per_scan = start.elapsed().as_secs_f64() * 1e6 / executions as f64;
                if round >= 5 { groups[usize::from(candidate)].push(per_scan); }
            }
        }
        println!("{label:<26} {:>13.2} {:>13.2}", median_us(&groups[0]), median_us(&groups[1]));
    }
}
println!("Native admission only. Each candidate batch includes the first classification and releases its owned text.");

PASS  rewrites use native admission; updates, commands, multiple statements and malformed text rejected


lifetime of one plan          current µs  candidate µs


1 execution                        34.83         34.96


32 same-text executions            33.73          1.04


32 alternating texts               33.52         17.57


Native admission only. Each candidate batch includes the first classification and releases its owned text.


In [20]:
// Candidate for bare MATCH: retain projection descriptors across DuckDB rebinds.
use std::{collections::VecDeque, ffi::{c_char, c_void}, sync::Arc};
type Field = (u8, Vec<u8>, Vec<u8>, Vec<u8>); // kind, label, property, output name
type Visit = extern "C" fn(*mut c_void, u8, *const c_char, *const c_char, *const c_char) -> bool;
type Project = unsafe extern "C" fn(*const c_char, Visit, *mut c_void) -> bool;
extern "C" fn collect_field(out: *mut c_void, kind: u8, l: *const c_char, p: *const c_char, n: *const c_char) -> bool {
    if out.is_null() || l.is_null() || p.is_null() || n.is_null() { return false; }
    // The provider borrows strings during this callback. Copy them before it returns.
    unsafe {
        let copy = |s| CStr::from_ptr(s).to_bytes().to_vec();
        (&mut *out.cast::<Vec<Field>>()).push((kind, copy(l), copy(p), copy(n)));
    }
    true
}
fn project_fields(project: Project, text: &CStr) -> Option<Arc<[Field]>> {
    let mut fields = Vec::<Field>::new();
    let ok = unsafe { project(text.as_ptr(), collect_field, (&mut fields as *mut Vec<Field>).cast()) };
    ok.then(|| Arc::from(fields))
}
#[derive(Default)]
struct ProjectionMemo {
    generation: Option<u64>,
    entries: VecDeque<(CString, Arc<[Field]>)>,
    parses: usize,
}
impl ProjectionMemo {
    fn lookup(&mut self, generation: u64, query: &CStr, project: Project) -> Option<Arc<[Field]>> {
        if self.generation != Some(generation) {
            self.entries.clear();
            self.generation = Some(generation);
        }
        if let Some((_, fields)) = self.entries.iter().find(|(key, _)| key.as_c_str() == query) {
            return Some(Arc::clone(fields));
        }
        self.parses += 1;
        let fields = project_fields(project, query)?;
        if query.to_bytes().len() <= 4096 {
            if self.entries.len() == 3 { self.entries.pop_front(); }
            self.entries.push_back((query.to_owned(), Arc::clone(&fields)));
        }
        Some(fields)
    }
}
{
    let project = unsafe { classifier.get::<Project>(b"probe_project\0")? };
    let project = *project; // classifier owns the library throughout this block
    let generation_id = generation.id();
    let cases = ["lookup", "lookup_miss", "fanout"];
    let texts: Vec<_> = cases.iter().map(|k| CString::new(snapshot["statements"][k].as_str().unwrap()).unwrap()).collect();
    let mut cache = ProjectionMemo::default();
    for text in &texts {
        assert_eq!(cache.lookup(generation_id, text, project).unwrap(), project_fields(project, text).unwrap());
    }
    let descriptor = cache.lookup(generation_id, &texts[1], project).unwrap();
    assert_eq!(&descriptor[..], &[(0, b"Person".to_vec(), b"PersonId".to_vec(), b"id".to_vec())]);
    assert_eq!(cache.parses, 3);
    assert!(cache.lookup(generation_id, c"CREATE (x:Person)", project).is_none());
    assert_eq!(cache.entries.len(), 3);
    assert_eq!(cache.lookup(generation_id + 1, &texts[1], project).unwrap(), descriptor);
    assert_eq!((cache.parses, cache.entries.len()), (5, 1));
    let worker = snapshot["workers"].as_array().unwrap().iter().find(|w| w["route"] == "qw_cypher").unwrap();
    let order: Vec<_> = worker["samples"].as_array().unwrap().iter().map(|s|
        cases.iter().position(|&k| s["case"] == k).unwrap()).collect();
    let mut times = [Vec::new(), Vec::new()];
    for round in 0..24 {
        for reuse in [round % 2 == 0, round % 2 != 0] {
            let mut memo = ProjectionMemo::default();
            let start = Instant::now();
            for &i in &order {
                let fields = if reuse { memo.lookup(generation_id, &texts[i], project) } else { project_fields(project, &texts[i]) };
                black_box(fields.expect("supported SF1 projection"));
            }
            let per_bind = start.elapsed().as_secs_f64() * 1e6 / order.len() as f64;
            if reuse { assert_eq!(memo.parses, cases.len()); }
            if round >= 4 { times[usize::from(reuse)].push(per_bind); }
        }
    }
    println!("{} mixed binds • {} distinct queries • cache starts empty in every batch", order.len(), cases.len());
    println!("Native projection, µs/bind: fresh {:.2}, bounded reuse {:.2}", median_us(&times[0]), median_us(&times[1]));
    println!("PASS  descriptor parity, rejection and generation change");
}

63 mixed binds • 3 distinct queries • cache starts empty in every batch


Native projection, µs/bind: fresh 37.46, bounded reuse 1.78


PASS  descriptor parity, rejection and generation change


()

### Which change helps bare `MATCH`?

**Start with projection reuse in the Ladybug adapter.** DuckDB can rebind a statement while the graph, dialect and query text all stay the same. In the local SF1 trace the adapter needed three native projection parses for 63 binds when it kept the three successful descriptions. The projection stage fell from 37.46 to 1.78 µs per bind. This is a candidate for the ordinary `MATCH … RETURN` surface, including exact repeats interleaved with other statements. The experiment measured one worker with three query texts. A production evaluation needs end-to-end timings, a byte budget, and concurrent and high-cardinality workloads.

| Component | Responsibility when reusing work |
| --- | --- |
| Client | A caller with a fixed alias, dialect and schema can prepare the documented `qw_query` on its DuckDB connection. The public client measured 0.224 ms p50 for the miss, versus 0.520 ms for bare Cypher and 0.091 ms for direct Ladybug on this Mac. |
| DuckDB grammar and binder | Bare Cypher and GQL retain their existing syntax. The grammar forwards the query. DuckDB reads the live session settings, owns the prepared plan and binds the result columns. The settings control measured 211 µs per lookup. A shortcut here must observe session graph and dialect changes before reusing a binding. |
| Quack Weave core | Resolve the registered provider and its current catalog. At scan start, check the current registration, file identity and DuckDB file-access policy. The live contract tests above exercise registration reset and disabled access after preparation. |
| Ladybug adapter | Own the bounded cache at its projection interface. Key it by exact emitted Cypher within a graph generation, keep successful projection descriptors and resolve their types against the current catalog. Preserve the independent native read classification and prepared-statement read-only check. |
| Ladybug engine | Prepare and execute the native query on its read-only session. The adapter continues to check the returned schema, including for zero rows. |

A plan-local read classification helps when the same bound plan runs repeatedly. Its cost per execution fell from 33.73 to 1.04 µs across 32 identical texts, including the initial parse. With one execution it cost about 35 µs either way. That makes it a separate candidate for long-lived prepared statements. The two component savings were measured independently and do not establish a combined end-to-end speedup.

### Repeated lookups on one configured connection

Run `configure` first. This experiment runs two full-SF1 workloads, first repeating one absent ID and then interleaving a present and an absent ID. Each worker opens one connection. DuckDB selects the graph and dialect before timing. The table includes native and ADBC calls through the complete Arrow fetch. The matched native baseline uses the same client preparation strategy as its Quack Weave counterpart.

Compare the two inferred scan rows to isolate the cost of live settings. Compare the parameterized rows to see what happens when an application keeps one `MATCH` handle and supplies different IDs. Each scan control also checks graph reset, dialect change and registration reset on a prepared handle after timing. The JSON snapshots contain setup times, exact statements, raw samples and probe outcomes.

In [24]:
// Run after configure. Each command publishes only after every result and probe passes.
{
    let python = std::env::var_os("QW_TRACKER_PYTHON").map(PathBuf::from)
        .or_else(|| ["build/tracker-venv/bin/python", "build/official/py312/bin/python"].iter()
            .map(|p| qw_root.join(p)).find(|p| p.is_file()))
        .ok_or("Set QW_TRACKER_PYTHON to the benchmark's pinned Python 3.12 environment")?;
    let out = qw_root.join("build/notebook-ladybug-miss");
    fs::create_dir_all(&out)?;
    let nonce = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH)?.as_nanos();
    for (label, cases) in [("miss-only", vec!["lookup_miss"]), ("mixed", vec!["lookup", "lookup_miss"])] {
        let path = out.join(format!("session-{label}-{nonce}.json"));
        let mut cmd = Command::new(&python);
        cmd.current_dir(&qw_root).args(["-B", "benchmarks/provider_comparison/tracker.py", "run",
            "--session-lookups", "--include-prepared", "--blocks", "4", "--runs", "50"]);
        for case in cases { cmd.arg("--case").arg(case); }
        let result = cmd.arg("--fixture").arg(&fixture).arg("--extension").arg(&extension)
            .arg("--out").arg(&path).output()?;
        if !result.status.success() { return Err(String::from_utf8_lossy(&result.stderr).into_owned().into()); }
        let report: serde_json::Value = serde_json::from_slice(&fs::read(&path)?)?;
        println!("{label} • lookup_miss • milliseconds • Python {} • {}", report["versions"]["python"], path.display());
        println!("{:<28} {:>4} {:>8} {:>8} {:>14}", "route", "n", "p50", "p95", "matched ratio");
        for row in report["summary"].as_array().unwrap().iter().filter(|r| r["case"] == "lookup_miss") {
            println!("{:<28} {:>4} {:>8.3} {:>8.3} {:>14}", row["route"].as_str().unwrap(), row["n"],
                row["p50_ms"].as_f64().unwrap(), row["p95_ms"].as_f64().unwrap(),
                row["qw_over_matched_direct"].as_f64().map(|r| format!("{r:.2}×")).unwrap_or_default());
        }
    }
}

miss-only • lookup_miss • milliseconds • Python "3.12.10" • /Users/kevintruong/project/quack-weave/build/notebook-ladybug-miss/session-miss-only-1795589184902292000.json


route                           n      p50      p95  matched ratio


direct_cypher                200    0.075    0.101               


direct_parameterized_cypher  200    0.013    0.018               


direct_prepared_cypher       200    0.012    0.017               


qw_cypher                    200    0.313    0.424          4.19×


... (13 outputs left out)

qw_prepared_literal_scan     200    0.188    0.230         11.90×


qw_prepared_match            200    0.353    0.568         22.39×


qw_prepared_session_scan     200    0.349    0.473         22.11×


()

On macOS arm64 with the pinned Python 3.12 client, a bare Cypher miss took 0.313 ms when repeated alone and 0.465 ms among hits. Direct Ladybug took 0.075 and 0.080 ms respectively. Query order changes the client-visible ratio from 4.19× to 5.78×, even though the application selects the graph once in both workloads.

The two inferred scans used the same query, provider, projection, ADBC handle lifetime and result checks. Reading live graph and dialect settings added 0.164 ms to the median miss-only lookup and 0.161 ms in mixed traffic. That difference includes rebinding work. The fixed scan still cost about 0.19 ms. A single parameterized `MATCH` handle also retained appreciable overhead (0.431 ms Quack Weave and 0.049 ms direct in mixed traffic).

Jev selected the allowed-tuple rule `data_integrity.mutually_consistent`. All six combinations of binding mode and session mutation produced the expected outcome in eight workers each. Z3 verified those observed facts (`sol_ffa64c8a8ce1416a`) and rejected a live handle returning the old graph’s hit after selection reset (`sol_8ad5153584d94e38`). The timings come from execution. A future binding-reuse candidate can be evaluated against both the latency rows and the post-timing behavior checks.

### When does graph execution outweigh initialization?

This experiment uses the full LSQB SF1 social graph, which adds friendship, tag, reply and location relationships to the creator edges used in the lookup experiment. Each workload starts a new session per block and runs repeatedly on that session. The benchmark records connection setup, the first query, and warm submissions through Arrow fetch. Ranked results must agree row for row. The four-hop reply count uses an explicit output schema; the other workloads use bare Cypher through Quack Weave.

Run `configure` first. Set `QW_PHASE_EXTENSION` to an optional diagnostic package to measure the native execute call inside Quack Weave. The ordinary release supplies the end-to-end comparison. [Benchmark and probe instructions](benchmarks/provider_comparison/tracker/README.md#complex-cypher-on-the-full-social-graph)

In [25]:
// Run the full SF1 workloads on independent clients and compare warm latency with setup.
{
    let python = std::env::var_os("QW_TRACKER_PYTHON").map(PathBuf::from)
        .or_else(|| ["build/tracker-venv/bin/python", "build/official/py312/bin/python"].iter()
            .map(|p| qw_root.join(p)).find(|p| p.is_file()))
        .ok_or("Set QW_TRACKER_PYTHON to the pinned benchmark interpreter")?;
    let nonce = std::time::SystemTime::now().duration_since(std::time::UNIX_EPOCH)?.as_nanos();
    let report_path = qw_root.join(format!("build/complex-sf1-{nonce}.json"));
    let mut cmd = Command::new(python);
    cmd.current_dir(&qw_root).args(["-B","benchmarks/provider_comparison/complex_cypher.py",
        "--blocks","6","--runs","8","--out"]).arg(&report_path).arg("--extension").arg(&extension);
    let local_probe = qw_root.join("build/complex-cypher-probe/quack_weave.duckdb_extension");
    let probe = std::env::var_os("QW_PHASE_EXTENSION").map(PathBuf::from)
        .or_else(|| local_probe.is_file().then_some(local_probe));
    if let Some(p) = probe { cmd.arg("--phase-extension").arg(p); }
    let result = cmd.output()?;
    if !result.status.success() { return Err(String::from_utf8_lossy(&result.stderr).into_owned().into()); }
    let report: serde_json::Value = serde_json::from_slice(&fs::read(&report_path)?)?;
    println!("Warm median, ms • 48 samples per workload and route");
    println!("{:<22} {:>9} {:>9} {:>11} {:>10} {:>10}", "workload","direct","QW","before exec","engine","engine %");
    for name in ["lookup_miss","creator_fanout","friends_of_friends","friends_message_tags","reply_countries"] {
        let row = &report["summary"][name];
        let phase = |f: &str| row["trace"][f].as_f64().map(|v| format!("{v:.3}")).unwrap_or_else(|| "—".into());
        println!("{name:<22} {:>9.3} {:>9.3} {:>11} {:>10} {:>10}", row["direct"]["p50_ms"].as_f64().unwrap(),
            row["qw"]["p50_ms"].as_f64().unwrap(), phase("before_native_ms"), phase("engine_ms"), phase("engine_share_pct"));
    }
    for route in ["direct","qw"] {
        let mut setup: Vec<_> = report["workers"].as_array().unwrap().iter().filter(|w| w["route"] == route)
            .map(|w| w["setup"]["total_ms"].as_f64().unwrap()).collect();
        setup.sort_by(f64::total_cmp);
        println!("One-time session setup ({route}): {:.2} ms median across {} sessions",
            (setup[(setup.len()-1)/2] + setup[setup.len()/2])/2.0, setup.len());
    }
    println!("Queries, startup phases, first requests, all samples and verified digests: {}", report_path.display());
}

Warm median, ms • 48 samples per workload and route


workload                  direct        QW before exec     engine   engine %


lookup_miss                0.081     0.306       0.289      0.008      2.707


creator_fanout             2.558     2.442       0.490      1.374     52.311


friends_of_friends        20.998    19.968       1.298     18.457     92.876


friends_message_tags     261.655   263.828       2.562    260.276     98.951


reply_countries          104.530   107.310       7.753     95.746     92.348


One-time session setup (direct): 24.75 ms median across 30 sessions


One-time session setup (qw): 90.37 ms median across 30 sessions


Queries, startup phases, first requests, all samples and verified digests: /Users/kevintruong/project/quack-weave/build/complex-sf1-1795590223249878000.json


()

On the tested arm64 Mac, Quack Weave reached a ready session in 90.37 ms versus 24.75 ms for direct Ladybug. Graph registration was the largest startup step at 58.00 ms median. Selecting the graph took 0.26 ms and setting the dialect took 0.12 ms. These are medians over 30 fresh sessions per client with warm filesystem caches.

For the three-hop lookup of tags on friends’ messages, the ordinary Quack Weave release took 263.83 ms versus 261.66 ms direct. The separate diagnostic run spent 2.56 ms before the native execute call and Ladybug reported 260.28 ms of engine execution. For friends of friends, the engine accounted for 93% of warm latency. For a point miss it accounted for just 3%. The four-hop reply count spent 92% in the engine and used an explicit output schema.

The biggest opportunity from reducing per-query setup is in short reads. Reusing a connection also spreads the 90 ms startup over all requests. For the three-hop query, most latency lies in the graph traversal itself. [Recorded samples](build/complex-sf1-1795590223249878000.json)

### Five different queries on one connection

Each client opened one connection per block and executed the five workloads sequentially on it. Quack Weave selected the graph and Cypher dialect once and reused one ADBC cursor. Each round contained all five queries in a seeded shuffle shared by both clients. Six blocks of ten rounds gave 50 submissions per connection and 48 measured warm rounds per client. Every returned table matched the earlier run, including row order for ranked queries.

| Median elapsed | Direct Ladybug | Quack Weave |
| --- | ---: | ---: |
| Set up the connection | 25.5 ms | 89.7 ms |
| Open the connection and finish the first five queries | 450.0 ms | 522.2 ms |
| Complete five queries on the warm connection | 386.4 ms | 388.3 ms |
| Set up the connection and execute all 50 queries | 3.939 s | 4.022 s |

The warm five-query round differed by 0.5%. Across the six paired blocks either client could finish first. Setup accounted for 2.2% of the 50-query Quack Weave session. The point miss still cost 1.002 ms through Quack Weave versus 0.216 ms direct. The three-hop tag traversal took 286.2 ms versus 284.9 ms and dominated the round.

Reproduce with the pinned benchmark Python environment using `benchmarks/provider_comparison/complex_cypher.py --mixed-session --out build/mixed-session.json`. The reported client times use the ordinary release. [Queries, order, phases and raw samples](build/mixed-session-sf1.flFzJh/results.json)